# RL Assisted MPC Markov Unified

Single supported notebook entrypoint for the consolidated polymer Markov TD3 supervisor.

In [1]:
from pathlib import Path
import os

from systems.polymer import get_polymer_notebook_defaults
from systems.polymer.data_io import canonical_baseline_path
from utils.notebook_setup import prepare_polymer_notebook_env, print_grouped_notebook_summary

NB = get_polymer_notebook_defaults("markov")

AGENT_KIND = NB["agent_kind"]
RUN_MODE = NB["run_mode"]
STYLE_PROFILE = NB["style_profile"]
SAVE_PDF = NB["save_pdf"]

POLYMER_DATA_DIR_OVERRIDE = NB["data_dir_override"]
POLYMER_RESULTS_DIR_OVERRIDE = NB["results_dir_override"]
RESULT_PREFIX_OVERRIDE = "td3_markov_disturb_step3_probe"
COMPARE_PREFIX_OVERRIDE = "disturb_compare_td3_markov_step3_probe"
BASELINE_MPC_PATH_OVERRIDE = NB["baseline_mpc_path_override"]
N_TESTS_OVERRIDE = 30
SET_POINTS_LEN_OVERRIDE = NB["set_points_len_override"]
WARM_START_OVERRIDE = NB["warm_start_override"]
TEST_CYCLE_OVERRIDE = NB["test_cycle_override"]
PLOT_START_EPISODE_OVERRIDE = NB["plot_start_episode_override"]
COMPARE_START_EPISODE_OVERRIDE = NB["compare_start_episode_override"]

# Optional one-off notebook overrides.
MAX_STEPS_OVERRIDE = None
DEBUG_VALIDATE_LIFTED_OVERRIDE = None
DEBUG_RUN_SHADOW_LS_OVERRIDE = None
LEGACY_MIMIC_STEP = "step2_nominal_reference_parity"
LEGACY_MIMIC_TD3_SEED_MODE = "deterministic"
LEGACY_MIMIC_SEED = 7

# Optional one-off release override.
# Keep this as None to reproduce the consolidated Step 3 surface.
# Set to 10 only for the later LS-warm-start tuning branch.
WARM_START_OVERRIDE = None

REPO_ROOT, DATA_DIR, RESULT_DIR = prepare_polymer_notebook_env(
    data_dir_override=POLYMER_DATA_DIR_OVERRIDE,
    results_dir_override=POLYMER_RESULTS_DIR_OVERRIDE,
)
os.chdir(REPO_ROOT)
RUN_PROFILE = NB["run_profiles"][(AGENT_KIND, RUN_MODE)]


In [2]:
import numpy as np

from Simulation.mpc import MpcSolverGeneral
from Simulation.system_functions import PolymerCSTR
from systems.polymer import POLYMER_SYSTEM_METADATA, load_polymer_system_data
from utils.helpers import apply_min_max
from utils.markov_runner_step3_probe import run_markov_correction_supervisor
from utils.plotting import compare_mpc_rl_from_dirs, plot_markov_correction_results
from utils.rewards import make_reward_fn_relative_QR


In [3]:
SYS = NB["system_setup"]
system_params = SYS["system_params"].copy()
system_design_params = SYS["design_params"].copy()
system_steady_state_inputs = SYS["ss_inputs"].copy()
delta_t = SYS["delta_t_hours"]

cstr_ss = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states = {"ss_inputs": cstr_ss.ss_inputs, "y_ss": cstr_ss.y_ss}

setpoint_y = SYS["setpoint_range_phys"].copy()
u_min = SYS["input_bounds"]["u_min"].copy()
u_max = SYS["input_bounds"]["u_max"].copy()

system_data = load_polymer_system_data(
    REPO_ROOT,
    steady_states=steady_states,
    setpoint_y=setpoint_y,
    u_min=u_min,
    u_max=u_max,
    n_inputs=2,
    data_override=POLYMER_DATA_DIR_OVERRIDE,
)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]

n_inputs = int(B_aug.shape[1])
y_sp_scenario_phys = SYS["rl_setpoints_phys"].copy()
y_sp_scenario = apply_min_max(y_sp_scenario_phys, data_min[n_inputs:], data_max[n_inputs:]) - apply_min_max(
    steady_states["y_ss"], data_min[n_inputs:], data_max[n_inputs:]
)


In [4]:
EPISODE_CFG = NB["episode_defaults"]
CTRL = NB["controller"]
TD3_CFG = NB["td3_agent"]
REWARD_CFG = NB["reward"]

n_tests = int(EPISODE_CFG["n_tests"] if N_TESTS_OVERRIDE is None else N_TESTS_OVERRIDE)
set_points_len = int(EPISODE_CFG["set_points_len"] if SET_POINTS_LEN_OVERRIDE is None else SET_POINTS_LEN_OVERRIDE)
warm_start = int(EPISODE_CFG["warm_start"] if WARM_START_OVERRIDE is None else WARM_START_OVERRIDE)
TEST_CYCLE = list(EPISODE_CFG["test_cycle"] if TEST_CYCLE_OVERRIDE is None else TEST_CYCLE_OVERRIDE)
PLOT_START_EPISODE = int(RUN_PROFILE["plot_start_episode"] if PLOT_START_EPISODE_OVERRIDE is None else PLOT_START_EPISODE_OVERRIDE)
COMPARE_START_EPISODE = int(RUN_PROFILE["compare_start_episode"] if COMPARE_START_EPISODE_OVERRIDE is None else COMPARE_START_EPISODE_OVERRIDE)
RESULT_PREFIX = RESULT_PREFIX_OVERRIDE or RUN_PROFILE["result_prefix"]
COMPARE_PREFIX = COMPARE_PREFIX_OVERRIDE or RUN_PROFILE["compare_prefix"]
BASELINE_MPC_PATH = Path(BASELINE_MPC_PATH_OVERRIDE).expanduser() if BASELINE_MPC_PATH_OVERRIDE else canonical_baseline_path(
    REPO_ROOT,
    RUN_MODE,
    data_override=POLYMER_DATA_DIR_OVERRIDE,
)

poles = SYS["observer_poles"].copy()
predict_h = CTRL["predict_h"]
cont_h = CTRL["cont_h"]
decision_interval = int(CTRL["decision_interval"])
Q1_penalty = CTRL["Q1_penalty"]
Q2_penalty = CTRL["Q2_penalty"]
R1_penalty = CTRL["R1_penalty"]
R2_penalty = CTRL["R2_penalty"]
basis_family = CTRL["basis_family"]
z_bound = float(CTRL["z_bound"])
prediction_window = int(CTRL["prediction_window"])
lambda_z = float(CTRL["lambda_z"])
s_pred_min = float(CTRL["s_pred_min"])
gain_drift_max = float(CTRL["gain_drift_max"])
nominal_cost_relative_tol = float(CTRL["nominal_cost_relative_tol"])
nominal_cost_absolute_tol = float(CTRL["nominal_cost_absolute_tol"])
nominal_solver_mode = str(CTRL.get("nominal_solver_mode", "state_space_shared"))
td3_seed = TD3_CFG.get("seed")
run_adaptive_ls = bool(CTRL["run_adaptive_ls"])
run_live_corrected_mpc = bool(CTRL["run_live_corrected_mpc"])
run_rl_proposal = bool(CTRL["run_rl_proposal"])
rl_fallback_to_ls = True
force_td3_execute = False
rl_store_executed_action_in_replay = bool(CTRL["rl_store_executed_action_in_replay"])
rl_save_agent_checkpoint = bool(CTRL["rl_save_agent_checkpoint"])
debug_validate_lifted = bool(CTRL["debug_validate_lifted"] if DEBUG_VALIDATE_LIFTED_OVERRIDE is None else DEBUG_VALIDATE_LIFTED_OVERRIDE)
debug_run_shadow_ls = bool(CTRL["debug_run_shadow_ls"] if DEBUG_RUN_SHADOW_LS_OVERRIDE is None else DEBUG_RUN_SHADOW_LS_OVERRIDE)
USE_SHIFTED_MPC_WARM_START = bool(CTRL["use_shifted_mpc_warm_start"])
observer_update_alignment = CTRL["observer_update_alignment"]

u_ss = apply_min_max(steady_states["ss_inputs"], data_min[:n_inputs], data_max[:n_inputs])
b_min = apply_min_max(SYS["input_bounds"]["u_min"], data_min[:n_inputs], data_max[:n_inputs]) - u_ss
b_max = apply_min_max(SYS["input_bounds"]["u_max"], data_min[:n_inputs], data_max[:n_inputs]) - u_ss

MPC_obj = MpcSolverGeneral(
    A_aug,
    B_aug,
    C_aug,
    Q_out=np.array([Q1_penalty, Q2_penalty], float),
    R_in=np.array([R1_penalty, R2_penalty], float),
    NP=predict_h,
    NC=cont_h,
)

reward_params, reward_fn = make_reward_fn_relative_QR(data_min, data_max, n_inputs=n_inputs, **REWARD_CFG)

print_grouped_notebook_summary(
    "Resolved Markov parameters",
    {
        "Paths": {
            "Repo root": REPO_ROOT,
            "Data dir": DATA_DIR,
            "Results dir": RESULT_DIR,
            "Baseline MPC": BASELINE_MPC_PATH,
        },
        "Run setup": {
            "Agent kind": AGENT_KIND,
            "Run mode": RUN_MODE,
            "TD3 seed": td3_seed,
            "n_tests": n_tests,
            "set_points_len": set_points_len,
            "warm_start": warm_start,
            "max_steps": MAX_STEPS_OVERRIDE,
            "plot_start_episode": PLOT_START_EPISODE,
            "compare_start_episode": COMPARE_START_EPISODE,
        },
        "Controller": {
            "predict_h": predict_h,
            "cont_h": cont_h,
            "decision_interval": decision_interval,
            "basis_family": basis_family,
            "z_bound": z_bound,
            "prediction_window": prediction_window,
            "gain_drift_max": gain_drift_max,
            "nominal_solver_mode": nominal_solver_mode,
            "force_td3_execute": force_td3_execute,
            "use_shifted_mpc_warm_start": USE_SHIFTED_MPC_WARM_START,
        },
        "Reward": reward_params,
        "Debug": {
            "debug_validate_lifted": debug_validate_lifted,
            "debug_run_shadow_ls": debug_run_shadow_ls,
        },
    },
)


Resolved Markov parameters

[Paths]
  Repo root           : C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\RL_assisted_MPC
  Data dir            : C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\RL_assisted_MPC\Polymer\Data
  Results dir         : C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\RL_assisted_MPC\Polymer\Results
  Baseline MPC        : C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\RL_assisted_MPC\Polymer\Data\mpc_results_dist.pickle

[Run setup]
  Agent kind          : td3
  Run mode            : disturb
  TD3 seed            : 7
  n_tests             : 30
  set_points_len      : 400
  warm_start          : 0
  max_steps           : None
  plot_start_episode  : 2
  compare_start_episode: 2

[Controller]
  predict_h           : 9
  cont_h              : 3
  decision_interval   : 1
  basis_family        : io_pair_gain
  z_bound             : 0.05
  prediction_window   : 20
  gain_drift_max      : 0.1
  nominal_solve

In [5]:
markov_cfg = {
    "agent_kind": AGENT_KIND,
    "run_mode": RUN_MODE,
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "warm_start": warm_start,
    "test_cycle": TEST_CYCLE,
    "predict_h": predict_h,
    "cont_h": cont_h,
    "decision_interval": decision_interval,
    "use_shifted_mpc_warm_start": USE_SHIFTED_MPC_WARM_START,
    "observer_update_alignment": observer_update_alignment,
    "basis_family": basis_family,
    "z_bound": z_bound,
    "prediction_window": prediction_window,
    "lambda_z": lambda_z,
    "s_pred_min": s_pred_min,
    "gain_drift_max": gain_drift_max,
    "nominal_solver_mode": nominal_solver_mode,
    "nominal_cost_relative_tol": nominal_cost_relative_tol,
    "nominal_cost_absolute_tol": nominal_cost_absolute_tol,
    "run_adaptive_ls": run_adaptive_ls,
    "run_live_corrected_mpc": run_live_corrected_mpc,
    "run_rl_proposal": run_rl_proposal,
    "rl_fallback_to_ls": rl_fallback_to_ls,
    "force_td3_execute": force_td3_execute,
    "rl_store_executed_action_in_replay": rl_store_executed_action_in_replay,
    "rl_save_agent_checkpoint": rl_save_agent_checkpoint,
    "debug_validate_lifted": debug_validate_lifted,
    "debug_run_shadow_ls": debug_run_shadow_ls,
    "nominal_qi": CTRL["nominal_qi"],
    "nominal_qs": CTRL["nominal_qs"],
    "nominal_ha": CTRL["nominal_ha"],
    "qi_change": CTRL["qi_change"],
    "qs_change": CTRL["qs_change"],
    "ha_change": CTRL["ha_change"],
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
    "b_min": b_min,
    "b_max": b_max,
    "td3_agent": TD3_CFG,
    "max_steps": MAX_STEPS_OVERRIDE,
    "legacy_mimic_step": LEGACY_MIMIC_STEP,
    "legacy_mimic_td3_seed_mode": LEGACY_MIMIC_TD3_SEED_MODE,
    "legacy_mimic_seed": LEGACY_MIMIC_SEED,
}

runtime_ctx = {
    "system_factory": lambda: PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t),
    "MPC_obj": MPC_obj,
    "steady_states": steady_states,
    "system_data": system_data,
    "data_min": data_min,
    "data_max": data_max,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "poles": poles,
    "y_sp_scenario": y_sp_scenario,
    "reward_fn": reward_fn,
    "reward_params": reward_params,
    "system_metadata": POLYMER_SYSTEM_METADATA,
    "delta_t": delta_t,
}

result_bundle = run_markov_correction_supervisor(markov_cfg=markov_cfg, runtime_ctx=runtime_ctx)
result_bundle["mpc_path_or_dir"] = BASELINE_MPC_PATH
result_bundle["agent_kind"] = AGENT_KIND
result_bundle["run_mode"] = RUN_MODE


Sub_Episode: 1 | avg. reward: -3.398689203581148 | accepted: 0.9625 | TD3 accepted: 0.34 | LS fallback: 0.6225 | nominal fallback: 0.02625 | avg z: [ 0.00545928  0.03907877  0.00510167 -0.01038814]


Sub_Episode: 2 | avg. reward: -4.425917577434314 | accepted: 1.0 | TD3 accepted: 0.42875 | LS fallback: 0.57125 | nominal fallback: 0.0 | avg z: [ 0.0044663   0.04258831  0.00676867 -0.01483937]


Sub_Episode: 3 | avg. reward: -4.394247967608741 | accepted: 1.0 | TD3 accepted: 0.3625 | LS fallback: 0.6375 | nominal fallback: 0.0 | avg z: [-0.00163734  0.04173539  0.00458578 -0.01635276]


Sub_Episode: 4 | avg. reward: -4.353625449008738 | accepted: 1.0 | TD3 accepted: 0.47 | LS fallback: 0.53 | nominal fallback: 0.0 | avg z: [ 0.00370244  0.04134099  0.00587163 -0.01579259]


Sub_Episode: 5 | avg. reward: -4.328299947131739 | accepted: 1.0 | TD3 accepted: 0.44375 | LS fallback: 0.55625 | nominal fallback: 0.0 | avg z: [ 0.00438163  0.0423873   0.00070262 -0.01478735]


...

Sub_Episode: 26 | avg. reward: -3.817444381317596 | accepted: 1.0 | TD3 accepted: 0.3525 | LS fallback: 0.6475 | nominal fallback: 0.0 | avg z: [ 0.0160412   0.03996778  0.00074257 -0.00217369]


Sub_Episode: 27 | avg. reward: -3.8109164848335784 | accepted: 0.97375 | TD3 accepted: 0.3125 | LS fallback: 0.66125 | nominal fallback: 0.02625 | avg z: [ 0.01562335  0.03517129 -0.00068504 -0.00732211]


Sub_Episode: 28 | avg. reward: -3.8164005362011393 | accepted: 0.875 | TD3 accepted: 0.255 | LS fallback: 0.62 | nominal fallback: 0.125 | avg z: [ 0.01560144  0.02510683 -0.00222707 -0.00340197]


Sub_Episode: 29 | avg. reward: -3.850852678417766 | accepted: 0.8525 | TD3 accepted: 0.26 | LS fallback: 0.5925 | nominal fallback: 0.1475 | avg z: [ 0.01755693  0.01187811 -0.00374221 -0.00097932]


Sub_Episode: 30 | avg. reward: -3.896936426594957 | accepted: 1.0 | TD3 accepted: 0.1825 | LS fallback: 0.8175 | nominal fallback: 0.0 | avg z: [ 2.15438929e-02 -5.21260492e-04  1.81546866e-05 -2.62785517e-03]


In [6]:
out_dir_rl = plot_markov_correction_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": RESULT_DIR,
        "prefix_name": RESULT_PREFIX,
        "start_episode": PLOT_START_EPISODE,
        "save_pdf": SAVE_PDF,
        "style_profile": STYLE_PROFILE,
        "save_agent_checkpoint": rl_save_agent_checkpoint,
        "agent_checkpoint_prefix": "td3_markov_agent",
        "s_pred_min": s_pred_min,
    },
)

out_dir_cmp = compare_mpc_rl_from_dirs(
    rl_dir=out_dir_rl,
    mpc_path_or_dir=BASELINE_MPC_PATH,
    reward_fn=reward_fn,
    directory=RESULT_DIR,
    prefix_name=COMPARE_PREFIX,
    compare_mode=RUN_PROFILE["compare_mode"],
    start_episode=COMPARE_START_EPISODE,
    n_inputs=n_inputs,
    save_pdf=SAVE_PDF,
    style_profile=STYLE_PROFILE,
)

print("RL result directory:", out_dir_rl)
print("Comparison directory:", out_dir_cmp)


Saved TD3 checkpoint to: C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\RL_assisted_MPC\Polymer\Results\td3_markov_disturb_step3_probe\20260511_015409\td3_markov_agent_20260511_015409.pkl


RL result directory: C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\RL_assisted_MPC\Polymer\Results\td3_markov_disturb_step3_probe\20260511_015409
Comparison directory: C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\RL_assisted_MPC\Polymer\Results\disturb_compare_td3_markov_step3_probe\20260511_015416


In [7]:
result_bundle["summary_metrics"]

{'agent_kind': 'td3',
 'run_mode': 'disturb',
 'nominal_solver_mode': 'lifted_g0_prototype',
 'force_td3_execute': False,
 'td3_seed': 7,
 'accepted_fraction': 0.9887916666666666,
 'td3_accepted_fraction': 0.3948333333333333,
 'ls_fallback_fraction': 0.5939583333333334,
 'nominal_fallback_fraction': 0.010833333333333334,
 'reward_mean': -4.029895784495305,
 'reward_final_episode': -3.896936426594957,
 'gain_drift_mean': 0.036968630184961424,
 'prediction_score_mean': 0.0323226933646016,
 'rl_replay_push_count': 23200,
 'rl_train_update_count': 23073}